# 한식 LoRA 학습 노트북 (SDXL-Turbo 기반)

AI Hub(aihub.or.kr)에서 다운로드한 한식 이미지로 LoRA를 학습합니다.
학습된 LoRA는 Google Drive에 저장되고, generate_images.ipynb에서 자동으로 불러옵니다.

## 사전 준비
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. AI Hub에서 다운로드한 한식 이미지를 Google Drive에 업로드
   - 경로: `/MyDrive/stockpipeline/lora_dataset/`
   - 권장: 단일 요리 사진 위주 30~100장
   - 형식: JPG, PNG (512px 이상)

## 예상 소요 시간
| 단계 | 시간 |
|------|------|
| 환경 설치 | 3분 |
| 모델 로드 | 2분 |
| 이미지 전처리 | 1분 |
| 학습 (1000 steps) | 60~90분 |
| 저장 & 테스트 | 5분 |


In [ ]:
# 1. 환경 설치 (한 번만 실행)
!pip install -q diffusers==0.30.0 transformers accelerate peft
!pip install -q bitsandbytes Pillow torch torchvision

from google.colab import drive, userdata
drive.mount('/content/drive')
print("\n✅ 환경 설치 완료")


In [ ]:
# 2. 설정 - 여기만 수정하면 됩니다
from pathlib import Path

GITHUB_REPO  = "stockpipeline/stockpipeline"
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

# 경로 설정
DATASET_DIR = "/content/drive/MyDrive/stockpipeline/lora_dataset"
OUTPUT_DIR  = "/content/drive/MyDrive/stockpipeline/lora_output"
WORK_DIR    = Path("/content/lora_work")

# 학습 설정
TRIGGER_WORD  = "koreanfood"   # 프롬프트에서 한식 스타일을 불러오는 키워드
TRAIN_STEPS   = 1000           # 데이터 30장: 800, 50장: 1000, 100장: 1500
LEARNING_RATE = 1e-4
LORA_RANK     = 16             # 4~32 (높을수록 특화도 ↑, 용량 ↑)
BATCH_SIZE    = 1              # T4 한계상 1 고정
SAVE_STEPS    = 500            # 중간 저장 주기

# 이미지 확인
from PIL import Image
import os

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

imgs = (list(Path(DATASET_DIR).glob("**/*.jpg")) +
        list(Path(DATASET_DIR).glob("**/*.jpeg")) +
        list(Path(DATASET_DIR).glob("**/*.png")))

print(f"발견된 이미지: {len(imgs)}장")
if len(imgs) == 0:
    print("❌ 이미지가 없습니다. DATASET_DIR 경로를 확인하세요.")
    print(f"   현재 경로: {DATASET_DIR}")
elif len(imgs) < 20:
    print("⚠️  20장 미만 - 품질이 낮을 수 있습니다")
else:
    print("✅ 학습 준비 완료")

# 샘플 이미지 확인
if imgs:
    from IPython.display import display
    sample = Image.open(imgs[0]).convert("RGB")
    sample.thumbnail((300, 300))
    print(f"\n샘플 이미지:")
    display(sample)


In [ ]:
# 3. 이미지 전처리 (512x512 + 캡션 생성)
import shutil

# 기존 작업 디렉토리 초기화
shutil.rmtree(str(WORK_DIR), ignore_errors=True)
WORK_DIR.mkdir(parents=True)

processed = 0
skipped = 0

for img_path in imgs:
    try:
        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        # 너무 작은 이미지 스킵
        if min(w, h) < 256:
            skipped += 1
            continue

        # 중앙 정사각형 크롭
        s = min(w, h)
        left = (w - s) // 2
        top  = (h - s) // 2
        img  = img.crop((left, top, left + s, top + s))
        img  = img.resize((512, 512), Image.LANCZOS)

        # 저장
        out = WORK_DIR / f"img_{processed:04d}.jpg"
        img.save(out, "JPEG", quality=95)

        # 캡션: .txt 파일이 있으면 그것을 사용, 없으면 기본 캡션 생성
        txt_src = img_path.with_suffix('.txt')
        if txt_src.exists():
            caption = f"{TRIGGER_WORD}, " + txt_src.read_text().strip()
        else:
            caption = (
                f"{TRIGGER_WORD}, korean food, "
                "single dish, food photography, "
                "white background, top-down view, "
                "isolated, commercial stock photo"
            )
        (WORK_DIR / f"img_{processed:04d}.txt").write_text(caption)
        processed += 1

    except Exception as e:
        skipped += 1

print(f"✅ 전처리 완료: {processed}장 (스킵: {skipped}장)")
print(f"   저장 위치: {WORK_DIR}")


In [ ]:
# 4. 모델 로드 + LoRA 설정
import torch
from diffusers import AutoPipelineForText2Image, DDPMScheduler
from peft import LoraConfig, get_peft_model

MODEL_ID = "stabilityai/sdxl-turbo"

print("SDXL-Turbo 로드 중... (2~3분)")
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to("cuda")

# 학습할 레이어 (UNet attention 레이어만)
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=[
        "to_q", "to_k", "to_v", "to_out.0",
        "add_q_proj", "add_k_proj", "add_v_proj",
    ],
    lora_dropout=0.05,
    bias="none",
)

pipe.unet = get_peft_model(pipe.unet, lora_config)
pipe.unet.print_trainable_parameters()

# 노이즈 스케줄러
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

print("\n✅ 모델 로드 완료")


In [ ]:
# 5. 학습
import time
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler

class FoodDataset(Dataset):
    def __init__(self, data_dir):
        self.imgs = sorted(Path(data_dir).glob("*.jpg"))
        self.txts = [Path(str(p).replace('.jpg', '.txt')) for p in self.imgs]
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]).convert("RGB")
        txt = self.txts[idx].read_text().strip()
        return self.transform(img), txt

dataset = FoodDataset(WORK_DIR)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)

optimizer = AdamW(pipe.unet.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scaler    = GradScaler()

# 텍스트/VAE는 고정
pipe.text_encoder.requires_grad_(False)
pipe.text_encoder_2.requires_grad_(False)
pipe.vae.requires_grad_(False)
pipe.unet.train()

print(f"학습 시작: {TRAIN_STEPS} steps | 데이터 {len(dataset)}장 | batch {BATCH_SIZE}")
print("-" * 60)

start       = time.time()
global_step = 0
loss_buf    = []

while global_step < TRAIN_STEPS:
    for imgs_b, captions in loader:
        if global_step >= TRAIN_STEPS:
            break

        imgs_b = imgs_b.to("cuda", dtype=torch.float16)

        with torch.no_grad():
            # 텍스트 인코딩
            def encode_text(tokenizer, text_encoder, texts):
                tokens = tokenizer(
                    texts, padding="max_length",
                    max_length=tokenizer.model_max_length,
                    truncation=True, return_tensors="pt"
                ).input_ids.to("cuda")
                return text_encoder(tokens, output_hidden_states=True)

            out1 = encode_text(pipe.tokenizer, pipe.text_encoder, captions)
            out2 = encode_text(pipe.tokenizer_2, pipe.text_encoder_2, captions)

            text_emb = torch.cat([
                out1.hidden_states[-2],
                out2.hidden_states[-2],
            ], dim=-1)
            pooled_emb = out2[0]

            # VAE 인코딩
            latents = pipe.vae.encode(imgs_b).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor

        # 노이즈 추가
        noise     = torch.randn_like(latents)
        timesteps = torch.randint(
            0, noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],), device="cuda"
        ).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # SDXL added cond
        bs = latents.shape[0]
        added_cond = {
            "text_embeds": pooled_emb,
            "time_ids": torch.tensor(
                [[1024, 1024, 0, 0, 1024, 1024]] * bs,
                device="cuda", dtype=torch.float16
            ),
        }

        # 순전파 + 역전파
        with autocast():
            pred = pipe.unet(
                noisy_latents, timesteps, text_emb,
                added_cond_kwargs=added_cond
            ).sample
            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(pipe.unet.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        global_step += 1
        loss_buf.append(loss.item())

        # 로그
        if global_step % 50 == 0:
            avg  = sum(loss_buf[-50:]) / len(loss_buf[-50:])
            ela  = time.time() - start
            eta  = ela / global_step * (TRAIN_STEPS - global_step)
            pct  = global_step / TRAIN_STEPS * 100
            print(f"  [{global_step:4d}/{TRAIN_STEPS}] {pct:5.1f}% | "
                  f"loss={avg:.4f} | 경과 {ela/60:.1f}분 | 남은 {eta/60:.1f}분")

        # 중간 저장
        if global_step % SAVE_STEPS == 0:
            ckpt = Path(OUTPUT_DIR) / f"checkpoint-{global_step}"
            pipe.unet.save_pretrained(str(ckpt))
            print(f"  → 중간 저장 완료: {ckpt.name}")

print(f"\n✅ 학습 완료! 총 {(time.time()-start)/60:.1f}분")
print(f"   최종 loss: {sum(loss_buf[-50:])/len(loss_buf[-50:]):.4f}")


In [ ]:
# 6. LoRA 저장 (Google Drive + GitHub)
import requests, base64

LORA_SAVE_PATH = Path(OUTPUT_DIR) / "korean_food_lora"
LORA_SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Drive에 저장
pipe.unet.save_pretrained(str(LORA_SAVE_PATH))
print(f"✅ Drive 저장 완료: {LORA_SAVE_PATH}")

# GitHub에도 push (50MB 이하면 가능, 보통 15~30MB)
safetensors_file = LORA_SAVE_PATH / "adapter_model.safetensors"
if not safetensors_file.exists():
    # peft 버전에 따라 파일명이 다를 수 있음
    candidates = list(LORA_SAVE_PATH.glob("*.safetensors")) + list(LORA_SAVE_PATH.glob("*.bin"))
    if candidates:
        safetensors_file = candidates[0]
        print(f"  파일명 확인: {safetensors_file.name}")

if safetensors_file.exists():
    file_size_mb = safetensors_file.stat().st_size / 1024 / 1024
    print(f"  파일 크기: {file_size_mb:.1f}MB")

    if file_size_mb < 50:
        with open(safetensors_file, "rb") as f:
            content = base64.b64encode(f.read()).decode()

        api_url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/lora/korean_food_lora.safetensors"
        headers = {"Authorization": f"Bearer {GITHUB_TOKEN}"}

        # 기존 파일 sha 확인
        existing = requests.get(api_url, headers=headers)
        payload  = {
            "message": "feat: update korean food LoRA weights",
            "content": content,
        }
        if existing.status_code == 200:
            payload["sha"] = existing.json()["sha"]

        res = requests.put(api_url, headers=headers, json=payload)
        if res.status_code in [200, 201]:
            print("✅ GitHub push 완료: lora/korean_food_lora.safetensors")
        else:
            print(f"⚠️  GitHub push 실패 ({res.status_code}). Drive에만 저장됩니다.")
    else:
        print(f"⚠️  파일이 50MB 초과 ({file_size_mb:.1f}MB). Drive에만 저장됩니다.")
        print("   generate_images.ipynb는 Drive에서 자동으로 로드합니다.")
else:
    print("⚠️  safetensors 파일 없음 - 저장 경로를 확인하세요")

print(f"\n완료! generate_images.ipynb를 실행하면 이 LoRA가 자동 적용됩니다.")


In [ ]:
# 7. 학습 결과 테스트
import requests
from IPython.display import display

# LoRA가 적용된 파이프라인으로 테스트
pipe.unet.eval()

config_data = requests.get(
    f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/config.json",
    headers={"Authorization": f"Bearer {GITHUB_TOKEN}"}
).json()
NEG = config_data.get("image", {}).get("negative_prompt",
    "multiple bowls, many small dishes, grid, tiled, collage")

test_cases = [
    (f"{TRIGGER_WORD} yukgaejang Korean spicy beef soup single bowl, top-down, white background", 42),
    (f"{TRIGGER_WORD} bibimbap Korean rice bowl single bowl, top-down, white background", 123),
    (f"{TRIGGER_WORD} kimchi-jjigae Korean kimchi stew single bowl, top-down, white background", 456),
]

print("LoRA 적용 테스트 결과:")
print("-" * 50)

for prompt, seed in test_cases:
    gen = torch.Generator(device="cuda").manual_seed(seed)
    img = pipe(
        prompt=prompt,
        negative_prompt=NEG,
        num_inference_steps=4,
        guidance_scale=1.5,
        width=1024, height=1024,
        generator=gen,
    ).images[0]
    print(f"\n{prompt.split(',')[0]}")
    display(img)

print("\n학습 결과가 마음에 드시면 파이프라인을 계속 진행하세요.")
print("아쉬우면 TRAIN_STEPS를 늘리거나 데이터를 더 추가하고 재학습해보세요.")
